In [2]:
import pickle
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import os
from scipy import stats
import matplotlib.pyplot as plt

In [3]:
# Set directory with WESAD data here
wesad_dir = Path("../Data/WESAD/")
# Empty list to store all the participants
subject_list = []
# Loop to open all the files
for i in wesad_dir.rglob("*.pkl"):
    subject_id = i.parent.name
    with open(i, "rb") as file:
        data = pickle.load(file, encoding='latin1')
        wrist_data = data['signal']['wrist']
        eda_signal = wrist_data['EDA'].flatten()
        #subject_id = data['label']
        # APPARENTLY the label is for 700hz and this needs to be recounted to match labels to 4hz accordingly
        label_700hz = data['label'].flatten()
        # 700/4=175 so we take chunks every 175 then label based on mode rule, I am open to suggestions for downsampling the labels
        n_chunks = len(label_700hz) // 175
        label_chunks = label_700hz[: n_chunks * 175].reshape(n_chunks, 175)
        label_4hz = stats.mode(label_chunks, axis=1).mode.flatten()
        # Rounding issue with getting mismatched vector lengths for concatenation
        min_len = min(len(eda_signal), len(label_4hz))
        # Build Dataframe
        df_subject = pd.DataFrame({
            'EDA': eda_signal[:min_len],
            'Label': label_4hz[:min_len]
            })
        df_subject.columns = [subject_id + '_EDA', subject_id + '_Label']
        #df_subject['Subject'] = subject_id
        subject_list.append(df_subject)

print(subject_list[1])

         S2_EDA  S2_Label
0      1.138257         0
1      1.125444         0
2      1.011405         0
3      1.033188         0
4      0.935807         0
...         ...       ...
24311  0.063052         0
24312  0.060489         0
24313  0.059208         0
24314  0.073303         0
24315  0.045113         0

[24316 rows x 2 columns]


In [4]:
subject_list[2]

,S3_EDA,S3_Label
0,2.248749,0
1,2.710001,0
2,2.554969,0
3,2.818907,0
4,2.630563,0
...,...,...
25967,0.302643,0
25968,0.301362,0
25969,0.301362,0
25970,0.301362,0


```
# Relabel numbers for easy human readable labels
final_df.loc[final_df['Label'] == 0, 'Label'] = 'Transient'
final_df.loc[final_df['Label'] == 1, 'Label'] = 'Baseline'
final_df.loc[final_df['Label'] == 2, 'Label'] = 'Stress'
final_df.loc[final_df['Label'] == 3, 'Label'] = 'Amusement'
final_df.loc[final_df['Label'] == 4, 'Label'] = 'Meditation'
final_df.loc[final_df['Label'] == 5, 'Label'] = 'Ignore'
final_df.loc[final_df['Label'] == 6, 'Label'] = 'Ignore'
final_df.loc[final_df['Label'] == 7, 'Label'] = 'Ignore'
final_df['Label'].unique()
```

We see each subject's EDA has different length.

In [5]:
for i in range(len(subject_list)):
    print(f"{len(subject_list[i])}")

25032
24316
25972
25692
23680
21984
20932
22524
21864
28284
20952
20892
22148
22192
21008


save the list that contains all EDA to a pickle file

In [6]:
with open('../Data/wesad_eda/wesad_E4_eda.pkl', 'wb') as file:
    pickle.dump(subject_list, file)